# Lesson 12.5 - LLM Inference Systems Engineering (Simulation Notebook)

## Learning Objectives
- Simulate queueing and latency for static vs continuous batching.
- Estimate throughput and tail latency effects.
- Connect KV-cache pressure to concurrency limits.

## Educational Scope and Production Extension Note
This notebook intentionally uses lightweight simulation/stub components for teaching. Treat outputs as instructional, not production-ready.

For production: replace simulated components with real services/models, add reliability controls, and instrument full observability.

In [1]:
from __future__ import annotations
import random
import numpy as np
import pandas as pd

np.random.seed(7)
random.seed(7)

## 1) Generate Synthetic Inference Traffic

In [2]:
n = 300
traffic = pd.DataFrame({
    'arrival_ms': np.cumsum(np.random.exponential(scale=40, size=n)).astype(int),
    'prompt_tokens': np.random.randint(80, 2000, size=n),
    'gen_tokens': np.random.randint(40, 600, size=n),
})
traffic.head()

,arrival_ms,prompt_tokens,gen_tokens
0,3,750,147
1,63,194,279
2,86,1936,258
3,138,242,189
4,290,164,278


## 2) Service Time Model

In [3]:
def service_time_ms(prompt_tokens, gen_tokens, prefill_rate=2500, decode_rate=120):
    prefill = (prompt_tokens / prefill_rate) * 1000
    decode = (gen_tokens / decode_rate) * 1000
    return prefill + decode

traffic['service_ms'] = traffic.apply(lambda r: service_time_ms(r.prompt_tokens, r.gen_tokens), axis=1)
traffic['service_ms'].describe()

count     300.000000
mean     3090.516000
std      1434.542963
min       497.400000
25%      1807.416667
50%      3156.200000
75%      4384.200000
max      5571.000000
Name: service_ms, dtype: float64

## 3) Static Batching Simulation

In [4]:
def simulate_static_batch(df, batch_size=8):
    rows = []
    i = 0
    current_time = 0.0
    while i < len(df):
        batch = df.iloc[i:i+batch_size].copy()
        if batch.empty:
            break
        start_time = max(current_time, batch.arrival_ms.min())
        batch_runtime = batch.service_ms.max()
        finish_time = start_time + batch_runtime
        for idx, r in batch.iterrows():
            rows.append((idx, finish_time - r.arrival_ms))
        current_time = finish_time
        i += batch_size
    return pd.Series(dict(rows)).sort_index()

lat_static = simulate_static_batch(traffic, batch_size=8)
lat_static.describe(percentiles=[0.5, 0.95, 0.99])

count       300.000000
mean      89384.156667
std       51175.116606
min        4024.200000
50%       88005.333333
95%      168349.183333
99%      177610.570000
max      177682.000000
dtype: float64

## 4) Continuous Batching Approximation

In [5]:
def simulate_continuous(df, concurrency=8):
    server_free = [0.0] * concurrency
    latencies = []
    for _, r in df.iterrows():
        j = int(np.argmin(server_free))
        start = max(server_free[j], r.arrival_ms)
        finish = start + r.service_ms
        server_free[j] = finish
        latencies.append(finish - r.arrival_ms)
    return pd.Series(latencies)

lat_cont = simulate_continuous(traffic, concurrency=8)
lat_cont.describe(percentiles=[0.5, 0.95, 0.99])

count       300.000000
mean      54712.030889
std       30427.625025
min        1525.000000
50%       56214.500000
95%      100660.586667
99%      105134.371333
max      106374.200000
dtype: float64

In [6]:
summary = pd.DataFrame({
    'mode': ['static_batch', 'continuous_batch'],
    'p50_ms': [lat_static.quantile(0.50), lat_cont.quantile(0.50)],
    'p95_ms': [lat_static.quantile(0.95), lat_cont.quantile(0.95)],
    'p99_ms': [lat_static.quantile(0.99), lat_cont.quantile(0.99)],
    'mean_ms': [lat_static.mean(), lat_cont.mean()],
})
summary.round(1)

,mode,p50_ms,p95_ms,p99_ms,mean_ms
0,static_batch,88005.3,168349.2,177610.6,89384.2
1,continuous_batch,56214.5,100660.6,105134.4,54712.0


## 5) KV Cache Pressure Estimation (Toy)

In [7]:
def kv_cache_gb(num_layers=32, hidden_size=4096, seq_len=4096, batch=8, bytes_per_elem=2):
    return (num_layers * batch * seq_len * hidden_size * 2 * bytes_per_elem) / (1024**3)

kv_df = pd.DataFrame({'batch': [4, 8, 16], 'seq_len': [2048, 4096, 8192]})
kv_df['kv_cache_gb'] = kv_df.apply(lambda r: kv_cache_gb(seq_len=int(r.seq_len), batch=int(r.batch)), axis=1)
kv_df.round(2)

,batch,seq_len,kv_cache_gb
0,4,2048,4.0
1,8,4096,16.0
2,16,8192,64.0


## Production Extension Checklist
- Replace simulated/stub components with production services or managed endpoints.
- Add structured logging, tracing IDs, and latency/error dashboards.
- Add input/output validation and policy/safety guardrails.
- Add retry/timeouts, rate limiting, and fallback behavior.
- Add offline/online evaluation gates before release.
- Add secrets management and environment-specific configuration.
- Add CI checks and smoke tests for critical paths.

## Case Studies & Exceptions
1. Support assistant with long context: context compression and caching often beat model-size escalation.
2. Content generation API: continuous batching improves utilization, but fairness controls are needed.
3. Exception: small predictable traffic can use simpler serving.

## Interview Questions & Answers
1. **Q:** Prefill vs decode? **A:** Prefill builds context cache; decode generates tokens iteratively.
2. **Q:** Why P95/P99? **A:** Tail latency determines user pain and SLA breaches.
3. **Q:** Static vs continuous batching? **A:** Fixed batch windows vs dynamic request admission.
4. **Q:** Why does KV cache matter? **A:** It is a major concurrency memory limit.
5. **Q:** First cost levers? **A:** Prompt size, output cap, and routing policy.
6. **Q:** Common anti-pattern? **A:** Hardware scaling before prompt/context optimization.
7. **Q:** Capacity planning input? **A:** Real token-length distributions.
8. **Q:** Why split queue wait and runtime? **A:** Better bottleneck diagnosis.
9. **Q:** Rollback lever? **A:** Route to last stable model/profile.
10. **Q:** One-line advice? **A:** LLM serving is queue plus memory engineering.